<a href="https://colab.research.google.com/github/seungil-kim/VehLatForceEstimation/blob/main/training_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/LSTM_LatForce

/content/drive/MyDrive/LSTM_LatForce


In [ ]:
# git clone

# !git clone https://github.com/seungil-kim/VehLatForceEstimation.git
# %cd VehLatForceEstimation

In [ ]:
## 데이터/라이브러리 불러오기 ##
import numpy as np
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os
from sklearn.metrics import mean_squared_error

In [ ]:
## 데이터셋 및 Scenario Split 불러오기 ##
data_path = Path("/content/drive/MyDrive/LSTM_LatForce/data")
# data_path = Path("/data")

mat_file = data_path / "LSTM_Dataset.mat"
split_mat_file = data_path / "LSTM_ScenarioSplit.mat"

if not mat_file.is_file():
    raise FileNotFoundError(f"Dataset 파일을 찾지 못했습니다: {mat_file}")

if not split_mat_file.is_file():
    raise FileNotFoundError(f"Scenario split 파일을 찾지 못했습니다: {split_mat_file}")

# Dataset은 기존 방식으로 로드
mat_data = loadmat(mat_file)
dataset = mat_data["Dataset"]

# Split은 MATLAB struct를 편하게 읽기 위해 별도 옵션 사용
split_mat_data = loadmat(
    split_mat_file,
    squeeze_me=True,
    struct_as_record=False
)

if "Split" not in split_mat_data:
    raise KeyError(
        "LSTM_ScenarioSplit.mat 내부에서 'Split' 변수를 찾지 못했습니다."
    )

split_data = split_mat_data["Split"]

print("Dataset keys:", mat_data.keys())
print("Dataset type:", type(dataset))
print("Dataset shape:", dataset.shape)
print("Split file:", split_mat_file)


Dataset keys: dict_keys(['__header__', '__version__', '__globals__', 'Dataset'])
Dataset type: <class 'numpy.ndarray'>
Dataset shape: (1, 1)
Split file: /content/drive/MyDrive/LSTM_LatForce/data/LSTM_ScenarioSplit.mat


In [ ]:
## 데이터셋 branch ##
scenario_names = dataset.dtype.names
input_names = dataset[0, 0][scenario_names[0]]["InputName"]
output_names = dataset[0, 0][scenario_names[0]]["OutputName"]

for name in scenario_names:
    scenario = dataset[0, 0][name]

    X = scenario["X"][0, 0]
    Y = scenario["Y"][0, 0]

    print(name, "X:", X.shape, "Y:", Y.shape)

DLC_Lturn_mu040_v040 X: (1301, 6) Y: (1301, 2)
DLC_Lturn_mu040_v060 X: (1064, 6) Y: (1064, 2)
DLC_Lturn_mu040_v080 X: (747, 6) Y: (747, 2)
DLC_Lturn_mu060_v040 X: (1301, 6) Y: (1301, 2)
DLC_Lturn_mu060_v060 X: (1064, 6) Y: (1064, 2)
DLC_Lturn_mu060_v080 X: (748, 6) Y: (748, 2)
DLC_Lturn_mu085_v040 X: (1301, 6) Y: (1301, 2)
DLC_Lturn_mu085_v060 X: (1064, 6) Y: (1064, 2)
DLC_Lturn_mu085_v080 X: (748, 6) Y: (748, 2)
DLC_Rturn_mu040_v040 X: (1301, 6) Y: (1301, 2)
DLC_Rturn_mu040_v060 X: (1064, 6) Y: (1064, 2)
DLC_Rturn_mu040_v080 X: (747, 6) Y: (747, 2)
DLC_Rturn_mu060_v040 X: (1301, 6) Y: (1301, 2)
DLC_Rturn_mu060_v060 X: (1064, 6) Y: (1064, 2)
DLC_Rturn_mu060_v080 X: (748, 6) Y: (748, 2)
DLC_Rturn_mu085_v040 X: (1301, 6) Y: (1301, 2)
DLC_Rturn_mu085_v060 X: (1064, 6) Y: (1064, 2)
DLC_Rturn_mu085_v080 X: (748, 6) Y: (748, 2)
ISO14791_Ay_Target_Lane_Change_Lturn_mu040_v040 X: (3401, 6) Y: (3401, 2)
ISO14791_Ay_Target_Lane_Change_Lturn_mu040_v060 X: (2201, 6) Y: (2201, 2)
ISO14791_Ay_Target

In [ ]:
## 데이터 검토
## pandas dataframe으로 변환하여 시각화
x_df = pd.DataFrame(X)
y_df = pd.DataFrame(Y)
# 컬럼 인덱
x_column_names = [name[0] for sublist in input_names[0,0] for name in sublist]
x_df.columns = x_column_names
y_column_names = [name[0] for sublist in output_names[0,0] for name in sublist]
y_df.columns = y_column_names

last_scenario_name = scenario_names[-1]
print(f"--- Last scenario: {last_scenario_name} ---")
print("\nInput (X) data head with column names:")
display(x_df.head())

print("\nOutput (Y) data head with column names:")
display(y_df.head())

--- Last scenario: Urban_Test_Area_mu085_vInternal ---

Input (X) data head with column names:


,vx_mps,vy_mps,ax_mps2,ay_mps2,yawrate_radps,steer_rad
0,16.665707,-0.000008,0.001760,-0.000192,-0.000012,-0.000003
1,16.665724,-0.000008,0.001667,-0.000203,-0.000012,-0.000003
2,16.665740,-0.000008,0.001580,-0.000214,-0.000013,-0.000003
3,16.665755,-0.000008,0.001503,-0.000224,-0.000013,-0.000003
4,16.665770,-0.000008,0.001435,-0.000234,-0.000014,-0.000003



Output (Y) data head with column names:


,Fyf_N,Fyr_N
0,-0.260748,-0.048662
1,-0.269989,-0.057001
2,-0.279043,-0.065165
3,-0.287913,-0.073155
4,-0.296608,-0.080975


In [ ]:
def seq_data(x, y, sequence_length):
    x_seq = []
    y_seq = []

    for i in range(len(x) - sequence_length + 1):
        x_seq.append(x[i : i + sequence_length])
        y_seq.append(y[i + sequence_length - 1])

    return np.array(x_seq), np.array(y_seq)

In [ ]:
## Train / Validation / Test 시나리오 분리 ##
# LSTM_ScenarioSplit.mat의 Split 구조체를 사용
# Split.Train, Split.Validation, Split.Test
# Split.StressTest가 있으면 별도로 유지

scenario_names = list(dataset.dtype.names)
scenario_name_set = set(scenario_names)

def matlab_cellstr_to_list(value):
    """MATLAB cellstr 또는 string 배열을 Python list[str]로 변환."""
    if isinstance(value, (str, np.str_)):
        return [str(value)]

    if isinstance(value, bytes):
        return [value.decode("utf-8")]

    values = np.asarray(value, dtype=object).ravel()
    scenario_list = []

    for item in values:
        # 1x1 ndarray가 중첩된 경우 내부 값 추출
        while isinstance(item, np.ndarray) and item.size == 1:
            item = item.reshape(-1)[0]

        if isinstance(item, bytes):
            item = item.decode("utf-8")

        scenario_list.append(str(item))

    return scenario_list


def get_split_field(split_struct, field_name, required=True):
    if not hasattr(split_struct, field_name):
        if required:
            raise KeyError(
                f"LSTM_ScenarioSplit.mat의 Split 구조체에 "
                f"'{field_name}' 필드가 없습니다."
            )
        return []

    return matlab_cellstr_to_list(getattr(split_struct, field_name))


train_scenarios = get_split_field(split_data, "Train")
val_scenarios = get_split_field(split_data, "Validation")
test_scenarios = get_split_field(split_data, "Test")

# StressTest는 split 파일에 있는 경우에만 별도 보관
stress_test_scenarios = get_split_field(
    split_data, "StressTest", required=False
)

split_dict = {
    "Train": train_scenarios,
    "Validation": val_scenarios,
    "Test": test_scenarios,
    "StressTest": stress_test_scenarios,
}

# 1. Dataset에 존재하지 않는 시나리오 검사
for split_name, split_scenarios in split_dict.items():
    missing_scenarios = sorted(set(split_scenarios) - scenario_name_set)

    if missing_scenarios:
        raise ValueError(
            f"[{split_name}] Dataset에 없는 시나리오가 있습니다:\n"
            + "\n".join(missing_scenarios)
        )

# 2. Split 간 중복 검사
scenario_to_split = {}

for split_name, split_scenarios in split_dict.items():
    for scenario_name in split_scenarios:
        if scenario_name in scenario_to_split:
            previous_split = scenario_to_split[scenario_name]

            raise ValueError(
                "Split 간 중복 시나리오가 있습니다:\n"
                f"{scenario_name}\n"
                f"- 기존 Split: {previous_split}\n"
                f"- 중복 Split: {split_name}"
            )

        scenario_to_split[scenario_name] = split_name

# 3. 어느 Split에도 포함되지 않은 시나리오 확인
unassigned_scenarios = sorted(
    scenario_name_set - set(scenario_to_split.keys())
)

print("==================================================")
print(f"Train scenarios       : {len(train_scenarios)}")
print(f"Validation scenarios  : {len(val_scenarios)}")
print(f"Test scenarios        : {len(test_scenarios)}")
print(f"Stress test scenarios : {len(stress_test_scenarios)}")
print(f"Unassigned scenarios  : {len(unassigned_scenarios)}")
print("==================================================")

print("\nTrain scenarios:")
print(train_scenarios)

print("\nValidation scenarios:")
print(val_scenarios)

print("\nTest scenarios:")
print(test_scenarios)

if stress_test_scenarios:
    print("\nStress test scenarios:")
    print(stress_test_scenarios)

if unassigned_scenarios:
    print("\n[Warning] Unassigned scenarios:")
    print(unassigned_scenarios)


Train scenarios       : 80
Validation scenarios  : 26
Test scenarios        : 21
Stress test scenarios : 6
Unassigned scenarios  : 0

Train scenarios:
['ISO_Steady_State_Circle_Lturn_mu040_v040', 'ISO_Steady_State_Circle_Lturn_mu040_v060', 'ISO_Steady_State_Circle_Lturn_mu060_v040', 'ISO_Steady_State_Circle_Lturn_mu060_v060', 'ISO_Steady_State_Circle_Lturn_mu060_v080', 'ISO_Steady_State_Circle_Lturn_mu085_v040', 'ISO_Steady_State_Circle_Lturn_mu085_v060', 'ISO_Steady_State_Circle_Lturn_mu085_v080', 'ISO_Steady_State_Circle_Rturn_mu040_v040', 'ISO_Steady_State_Circle_Rturn_mu040_v060', 'ISO_Steady_State_Circle_Rturn_mu060_v040', 'ISO_Steady_State_Circle_Rturn_mu060_v060', 'ISO_Steady_State_Circle_Rturn_mu060_v080', 'ISO_Steady_State_Circle_Rturn_mu085_v040', 'ISO_Steady_State_Circle_Rturn_mu085_v060', 'ISO_Steady_State_Circle_Rturn_mu085_v080', 'Mcity_Outside_Loop_mu040_v040', 'Mcity_Outside_Loop_mu060_v040', 'Mcity_Outside_Loop_mu060_v060', 'Mcity_Outside_Loop_mu085_v040', 'On_Center_S

In [ ]:
def get_scenario_data(dataset, name):
    scenario = dataset[0, 0][name]

    X = scenario["X"][0, 0].astype(np.float32)
    Y = scenario["Y"][0, 0].astype(np.float32)

    time = scenario["Time"][0, 0]
    sim_time = float(np.max(time))

    return X, Y, sim_time

In [ ]:
# Train 시나리오 원본 데이터를 모두 모음
X_train_raw_all = []
Y_train_raw_all = []

for name in train_scenarios:
    X, Y, _ = get_scenario_data(dataset, name)

    X_train_raw_all.append(X)
    Y_train_raw_all.append(Y)

X_train_raw_all = np.concatenate(X_train_raw_all, axis=0)
Y_train_raw_all = np.concatenate(Y_train_raw_all, axis=0)
print("X_train_shape: ", {X_train_raw_all.shape})
print("Y_train_shape: ", {Y_train_raw_all.shape})

# 입력 X와 출력 Y는 별도 scaler 사용
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

x_scaler.fit(X_train_raw_all)
y_scaler.fit(Y_train_raw_all)

X_train_shape:  {(627842, 6)}
Y_train_shape:  {(627842, 2)}


MinMaxScaler()

In [ ]:
def make_sequence_dataset(scenario_list, dataset_name, sequence_length):
    x_list = []
    y_list = []

    for name in scenario_list:
        X, Y, sim_time = get_scenario_data(dataset, name)

        X_scaled = x_scaler.transform(X)
        Y_scaled = y_scaler.transform(Y)

        x_seq, y_seq = seq_data(X_scaled, Y_scaled, sequence_length)

        x_list.append(x_seq)
        y_list.append(y_seq)

        print(
            f"[{dataset_name}] {name} -> "
            f"X_seq: {x_seq.shape}, "
            f"Y_seq: {y_seq.shape}, "
            f"Sim_Time: {sim_time:.2f}"
        )

    X_out = np.concatenate(x_list, axis=0)
    Y_out = np.concatenate(y_list, axis=0)

    return X_out, Y_out

In [ ]:
sequence_length = 5

x_train_seq, y_train_seq = make_sequence_dataset(train_scenarios, "Train", sequence_length)

x_val_seq, y_val_seq = make_sequence_dataset(sorted(val_scenarios),"Validation", sequence_length)

print("\n================================")
print("Train X:", x_train_seq.shape)
print("Train Y:", y_train_seq.shape)

print("\nValidation X:", x_val_seq.shape)
print("Validation Y:", y_val_seq.shape)

[Train] ISO_Steady_State_Circle_Lturn_mu040_v040 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu040_v060 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu060_v040 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu060_v060 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu060_v080 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu085_v040 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu085_v060 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Lturn_mu085_v080 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] ISO_Steady_State_Circle_Rturn_mu040_v040 -> X_seq: (19896, 5, 6), Y_seq: (19896, 2), Sim_Time: 200.99
[Train] IS

In [ ]:
## GPU 사용 여부 확인 (Cuda)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("▶device: ", device)

# 텐서로 변환
x_train_seq = torch.as_tensor(x_train_seq, dtype=torch.float32).to(device)
y_train_seq = torch.as_tensor(y_train_seq, dtype=torch.float32).to(device)
x_val_seq = torch.as_tensor(x_val_seq, dtype=torch.float32).to(device)
y_val_seq = torch.as_tensor(y_val_seq, dtype=torch.float32).to(device)

print("[Train]", x_train_seq.size())
print("[Train]", y_train_seq.size())
print("[Validation]", x_val_seq.size())
print("[Validation]", y_val_seq.size())

▶device:  cuda:0
[Train] torch.Size([627522, 5, 6])
[Train] torch.Size([627522, 2])
[Validation] torch.Size([78610, 5, 6])
[Validation] torch.Size([78610, 2])


In [ ]:
## 배치 형태로 변경 ##
train_dataset = torch.utils.data.TensorDataset(x_train_seq, y_train_seq)
val_dataset = torch.utils.data.TensorDataset(x_val_seq, y_val_seq)

batch_size = 128
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size = batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size = batch_size)

In [ ]:
## LSTM 모델 생성 ##
#기본 하이퍼 파라미터 설정
input_size = x_train_seq.size(2)
num_layers = 2
hidden_size = 8

os.makedirs("models", exist_ok=True)

In [ ]:
class LSTM(nn.Module):
  def __init__(self, input_size, hidden_size, sequence_length, num_layers, device):
    super(LSTM, self).__init__()
    self.device = device
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first = True)
    self.fc = nn.Linear(hidden_size*sequence_length ,2) # Changed output from 1 to 2

  def forward(self, x):
    h0 = torch.zeros(self.num_layers, x.size()[0], self.hidden_size).to(self.device)
    c0 = torch.zeros(self.num_layers, x.size()[0], self.hidden_size).to(self.device)
    out, _ = self.lstm(x,(h0,c0))
    out = out.reshape(out.shape[0], -1)
    out = self.fc(out)
    return out

In [ ]:
## LSTM 모델 불러오기 ##
model = LSTM(input_size = input_size,
             hidden_size=hidden_size,
             sequence_length=sequence_length,
             num_layers = num_layers,
             device=device).to(device)


In [ ]:
## 손실함수 및 최적화 방법 정의 ##
criterion = nn.MSELoss()
num_epochs =401
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
train_loss_graph = []
val_loss_graph = []

best_val_loss = float("inf")
best_epoch = -1

early_stopping_patience = 20
min_delta = 0.0
patience_counter = 0

last_epoch = -1

for epoch in range(num_epochs):

    # =====================================
    # 1. Train
    # =====================================
    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for seq, target in train_loader:

        seq = seq.to(device)
        target = target.to(device)

        out = model(seq)
        loss = criterion(out, target)

        optimizer.zero_grad()
        loss.backward()

        # 필요 시 기울기 폭주 방지
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        batch_size_now = seq.size(0)
        train_loss_sum += loss.item() * batch_size_now
        train_sample_count += batch_size_now

    train_loss = train_loss_sum / train_sample_count
    train_loss_graph.append(train_loss)

    # =====================================
    # 2. Validation
    # =====================================
    model.eval()

    val_loss_sum = 0.0
    val_sample_count = 0

    with torch.no_grad():
        for seq, target in val_loader:

            seq = seq.to(device)
            target = target.to(device)

            out = model(seq)
            loss = criterion(out, target)

            batch_size_now = seq.size(0)
            val_loss_sum += loss.item() * batch_size_now
            val_sample_count += batch_size_now

    val_loss = val_loss_sum / val_sample_count
    val_loss_graph.append(val_loss)

    # =====================================
    # 3. Best model 저장 및 Early stopping
    # =====================================
    is_improved = val_loss < (best_val_loss - min_delta)

    if is_improved:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_loss_graph": train_loss_graph.copy(),
            "val_loss_graph": val_loss_graph.copy(),
        }, "models/lstm_best_model.pth")

    else:
        patience_counter += 1

    # =====================================
    # 4. 학습 상태 출력
    # =====================================
    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(
            f"[Epoch {epoch:03d}] "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Best Val Loss: {best_val_loss:.6f} | "
            f"Patience: {patience_counter}/{early_stopping_patience}"
        )

    # =====================================
    # 5. Early stopping 판단
    # =====================================
    if patience_counter >= early_stopping_patience:
        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch}, "
            f"Best val loss: {best_val_loss:.6f}"
        )
        last_epoch = epoch
        break

    last_epoch = epoch

# 마지막 epoch 모델 저장
torch.save({
    "epoch": last_epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_loss": train_loss_graph[-1],
    "val_loss": val_loss_graph[-1],
}, "models/lstm_last_model.pth")

# 최종 평가 전 Best model 복원
best_checkpoint = torch.load(
    "models/lstm_best_model.pth",
    map_location=device
)

model.load_state_dict(best_checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("\n================================")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print("Best model loaded: models/lstm_best_model.pth")

[Epoch 000] Train Loss: 0.000829 | Val Loss: 0.000342 | Best Val Loss: 0.000342 | Patience: 0/20
[Epoch 010] Train Loss: 0.000010 | Val Loss: 0.000099 | Best Val Loss: 0.000099 | Patience: 0/20
[Epoch 020] Train Loss: 0.000006 | Val Loss: 0.000047 | Best Val Loss: 0.000047 | Patience: 0/20
[Epoch 030] Train Loss: 0.000004 | Val Loss: 0.000031 | Best Val Loss: 0.000031 | Patience: 0/20
[Epoch 040] Train Loss: 0.000003 | Val Loss: 0.000025 | Best Val Loss: 0.000025 | Patience: 0/20
[Epoch 050] Train Loss: 0.000003 | Val Loss: 0.000019 | Best Val Loss: 0.000019 | Patience: 0/20
[Epoch 060] Train Loss: 0.000002 | Val Loss: 0.000018 | Best Val Loss: 0.000018 | Patience: 0/20
[Epoch 070] Train Loss: 0.000002 | Val Loss: 0.000017 | Best Val Loss: 0.000016 | Patience: 2/20
[Epoch 080] Train Loss: 0.000002 | Val Loss: 0.000016 | Best Val Loss: 0.000015 | Patience: 4/20
[Epoch 090] Train Loss: 0.000002 | Val Loss: 0.000014 | Best Val Loss: 0.000014 | Patience: 1/20
[Epoch 100] Train Loss: 0.0000

In [ ]:
## loss 그래프
os.makedirs("results", exist_ok=True)

plt.figure(figsize=(9, 5))

epochs = range(1, len(train_loss_graph) + 1)

plt.plot(epochs, train_loss_graph, label="Train Loss")
plt.plot(epochs, val_loss_graph, label="Validation Loss")

# validation loss가 가장 낮은 epoch 표시
plt.axvline(
    best_epoch + 1,
    linestyle="--",
    label=f"Best Epoch = {best_epoch + 1}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("LSTM Training and Validation Loss")
plt.grid(True)
plt.legend()
plt.savefig(
    f"results/Loss.png",
    dpi=300
)

NameError: name 'os' is not defined

In [ ]:
## loss-log 그래프

plt.figure(figsize=(9, 5))

plt.semilogy(train_loss_graph, label="Train Loss")
plt.semilogy(val_loss_graph, label="Validation Loss")

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss (log scale)")
plt.title("LSTM Training and Validation Loss")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()

plt.savefig(
    f"results/Loss_log.png",
    dpi=300
)
plt.show()

In [ ]:
def calc_force_metrics(true_force, pred_force, force_name):
    error = pred_force - true_force

    rmse = np.sqrt(np.mean(error**2))
    mae = np.mean(np.abs(error))
    max_abs_error = np.max(np.abs(error))

    peak_to_peak = np.max(true_force) - np.min(true_force)
    nrmse_percent = 100 * rmse / peak_to_peak

    print(f"\n[{force_name}]")
    print(f"RMSE      : {rmse:.2f} N")
    print(f"MAE       : {mae:.2f} N")
    print(f"Max Error : {max_abs_error:.2f} N")
    print(f"NRMSE     : {nrmse_percent:.2f} %")

    return rmse, mae, max_abs_error, nrmse_percent

In [ ]:
checkpoint = torch.load(
    "models/lstm_best_model.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Loaded best epoch:", checkpoint["epoch"])
print("Loaded best val loss:", checkpoint["val_loss"])

In [ ]:
# MATLAB struct 내부 Time 데이터를 1차원 float 배열로 변환
def get_time_array(time_raw):
    time_data = time_raw

    # 1x1 object array가 중첩된 경우 내부 값 추출
    while (
        isinstance(time_data, np.ndarray)
        and time_data.dtype == object
        and time_data.size == 1
    ):
        time_data = time_data.flat[0]

    return np.asarray(time_data, dtype=np.float32).squeeze()


# 한 시나리오에 대해 추론 수행
def predict_scenario(dataset, scenario_name, model,
                     x_scaler, y_scaler,
                     sequence_length, device):

    scenario = dataset[0, 0][scenario_name]

    # 원본 데이터
    X_raw = scenario["X"][0, 0].astype(np.float32)
    Y_raw = scenario["Y"][0, 0].astype(np.float32)
    time = get_time_array(scenario["Time"])

    # 비교용 UKF 횡력
    try:
        ukf_raw = (
            scenario["UKF"][0, 0]
            .astype(np.float32)
        )
    except (KeyError, IndexError, TypeError, ValueError) as e:
        raise KeyError(
            f"{scenario_name}: Comparison.UKF 데이터를 읽을 수 없습니다. "
            "MATLAB Dataset에 Comparison.UKF 필드가 있는지 확인하세요."
        ) from e

    # 데이터 형상 확인
    if X_raw.ndim != 2 or Y_raw.ndim != 2 or ukf_raw.ndim != 2:
        raise ValueError(
            f"{scenario_name}: X, Y, UKF 데이터는 모두 2차원 행렬이어야 합니다."
        )

    if Y_raw.shape[1] != 2 or ukf_raw.shape[1] != 2:
        raise ValueError(
            f"{scenario_name}: Y와 UKF는 [Fyf, Fyr] 형태의 N×2 행렬이어야 합니다."
        )

    if not (
        len(X_raw) == len(Y_raw) == len(ukf_raw) == len(time)
    ):
        raise ValueError(
            f"{scenario_name}: X, Y, UKF, Time 길이가 일치하지 않습니다.\n"
            f"X={len(X_raw)}, Y={len(Y_raw)}, "
            f"UKF={len(ukf_raw)}, Time={len(time)}"
        )

    if len(X_raw) < sequence_length:
        raise ValueError(
            f"{scenario_name}: sequence_length={sequence_length}보다 "
            "데이터 길이가 짧습니다."
        )

    # Train 데이터 기준 scaler 적용
    X_scaled = x_scaler.transform(X_raw)

    # 학습의 seq_data()와 동일한 sequence 정의
    # 입력  [i : i + sequence_length]
    # 정답  i + sequence_length - 1
    x_seq, _ = seq_data(X_scaled, Y_raw, sequence_length)
    x_seq = x_seq.astype(np.float32)

    # LSTM 추론
    x_tensor = torch.as_tensor(
        x_seq,
        dtype=torch.float32,
        device=device
    )

    model.eval()

    with torch.no_grad():
        pred_scaled = model(x_tensor).cpu().numpy()

    # 정규화 영역 -> 실제 횡력 단위 복원
    pred_force = y_scaler.inverse_transform(pred_scaled)

    # seq_data()의 target index와 시간축 정렬
    start_idx = sequence_length - 1

    time_aligned = time[start_idx:]
    true_force = Y_raw[start_idx:]
    ukf_force = ukf_raw[start_idx:]

    if not (
        len(time_aligned)
        == len(true_force)
        == len(pred_force)
        == len(ukf_force)
    ):
        raise RuntimeError(
            f"{scenario_name}: 예측값/정답/UKF/시간축 길이가 일치하지 않습니다."
        )

    return time_aligned, true_force, pred_force, ukf_force


def evaluate_scenario_list(
    scenario_list,
    split_name,
    metrics_path,
    dataset,
    model,
    x_scaler,
    y_scaler,
    sequence_length,
    device
):
    """시나리오 목록의 LSTM 및 UKF 횡력 추정 성능을 TXT, 표, 그래프로 출력."""

    os.makedirs("results", exist_ok=True)

    metric_rows = []

    with open(metrics_path, "w", encoding="utf-8") as f:
        f.write("Lateral Force Estimation Metrics\n")
        f.write(f"Split: {split_name}\n")
        f.write("=" * 70 + "\n")

    for scenario_name in scenario_list:

        time_eval, true_force, pred_force, ukf_force = predict_scenario(
            dataset=dataset,
            scenario_name=scenario_name,
            model=model,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            sequence_length=sequence_length,
            device=device
        )

        # -------------------------------------------------
        # LSTM 오차 및 지표
        # -------------------------------------------------
        lstm_error_front = pred_force[:, 0] - true_force[:, 0]
        lstm_error_rear = pred_force[:, 1] - true_force[:, 1]

        lstm_rmse_front = np.sqrt(np.mean(lstm_error_front ** 2))
        lstm_rmse_rear = np.sqrt(np.mean(lstm_error_rear ** 2))

        lstm_mae_front = np.mean(np.abs(lstm_error_front))
        lstm_mae_rear = np.mean(np.abs(lstm_error_rear))

        lstm_max_error_front = np.max(np.abs(lstm_error_front))
        lstm_max_error_rear = np.max(np.abs(lstm_error_rear))

        # -------------------------------------------------
        # UKF 오차 및 지표
        # -------------------------------------------------
        ukf_error_front = ukf_force[:, 0] - true_force[:, 0]
        ukf_error_rear = ukf_force[:, 1] - true_force[:, 1]

        ukf_rmse_front = np.sqrt(np.mean(ukf_error_front ** 2))
        ukf_rmse_rear = np.sqrt(np.mean(ukf_error_rear ** 2))

        ukf_mae_front = np.mean(np.abs(ukf_error_front))
        ukf_mae_rear = np.mean(np.abs(ukf_error_rear))

        ukf_max_error_front = np.max(np.abs(ukf_error_front))
        ukf_max_error_rear = np.max(np.abs(ukf_error_rear))

        print(f"\n[{split_name}] {scenario_name}")

        print("[LSTM]")
        print(f"Front RMSE      : {lstm_rmse_front:.2f} N")
        print(f"Front MAE       : {lstm_mae_front:.2f} N")
        print(f"Front Max Error : {lstm_max_error_front:.2f} N")
        print(f"Rear RMSE       : {lstm_rmse_rear:.2f} N")
        print(f"Rear MAE        : {lstm_mae_rear:.2f} N")
        print(f"Rear Max Error  : {lstm_max_error_rear:.2f} N")

        print("[UKF]")
        print(f"Front RMSE      : {ukf_rmse_front:.2f} N")
        print(f"Front MAE       : {ukf_mae_front:.2f} N")
        print(f"Front Max Error : {ukf_max_error_front:.2f} N")
        print(f"Rear RMSE       : {ukf_rmse_rear:.2f} N")
        print(f"Rear MAE        : {ukf_mae_rear:.2f} N")
        print(f"Rear Max Error  : {ukf_max_error_rear:.2f} N")

        metric_rows.append({
            "Scenario": scenario_name,

            "LSTM_Front_RMSE_N": lstm_rmse_front,
            "LSTM_Front_MAE_N": lstm_mae_front,
            "LSTM_Front_MaxError_N": lstm_max_error_front,
            "LSTM_Rear_RMSE_N": lstm_rmse_rear,
            "LSTM_Rear_MAE_N": lstm_mae_rear,
            "LSTM_Rear_MaxError_N": lstm_max_error_rear,

            "UKF_Front_RMSE_N": ukf_rmse_front,
            "UKF_Front_MAE_N": ukf_mae_front,
            "UKF_Front_MaxError_N": ukf_max_error_front,
            "UKF_Rear_RMSE_N": ukf_rmse_rear,
            "UKF_Rear_MAE_N": ukf_mae_rear,
            "UKF_Rear_MaxError_N": ukf_max_error_rear,
        })

        with open(metrics_path, "a", encoding="utf-8") as f:
            f.write(f"\nScenario: {scenario_name}\n")
            f.write("-" * 70 + "\n")

            f.write("[LSTM]\n")
            f.write(f"Front RMSE      : {lstm_rmse_front:.2f} N\n")
            f.write(f"Front MAE       : {lstm_mae_front:.2f} N\n")
            f.write(f"Front Max Error : {lstm_max_error_front:.2f} N\n")
            f.write(f"Rear RMSE       : {lstm_rmse_rear:.2f} N\n")
            f.write(f"Rear MAE        : {lstm_mae_rear:.2f} N\n")
            f.write(f"Rear Max Error  : {lstm_max_error_rear:.2f} N\n")

            f.write("\n[UKF]\n")
            f.write(f"Front RMSE      : {ukf_rmse_front:.2f} N\n")
            f.write(f"Front MAE       : {ukf_mae_front:.2f} N\n")
            f.write(f"Front Max Error : {ukf_max_error_front:.2f} N\n")
            f.write(f"Rear RMSE       : {ukf_rmse_rear:.2f} N\n")
            f.write(f"Rear MAE        : {ukf_mae_rear:.2f} N\n")
            f.write(f"Rear Max Error  : {ukf_max_error_rear:.2f} N\n")

        # -------------------------------------------------
        # 전륜 횡력 그래프
        # -------------------------------------------------
        plt.figure(figsize=(10, 4))

        plt.plot(time_eval, true_force[:, 0], label="Ground Truth")
        plt.plot(time_eval, ukf_force[:, 0], label="UKF Estimation")
        plt.plot(time_eval, pred_force[:, 0], label="LSTM Prediction")

        plt.xlabel("Time [s]")
        plt.ylabel("Front Lateral Force [N]")
        plt.title(f"{split_name} | {scenario_name} - Front Lateral Force")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            f"results/{split_name}_{scenario_name}_front_lateral_force.png",
            dpi=300
        )
        plt.show()

        # -------------------------------------------------
        # 후륜 횡력 그래프
        # -------------------------------------------------
        plt.figure(figsize=(10, 4))

        plt.plot(time_eval, true_force[:, 1], label="Ground Truth")
        plt.plot(time_eval, ukf_force[:, 1], label="UKF Estimation")
        plt.plot(time_eval, pred_force[:, 1], label="LSTM Prediction")

        plt.xlabel("Time [s]")
        plt.ylabel("Rear Lateral Force [N]")
        plt.title(f"{split_name} | {scenario_name} - Rear Lateral Force")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            f"results/{split_name}_{scenario_name}_rear_lateral_force.png",
            dpi=300
        )
        plt.show()

    metrics_df = pd.DataFrame(metric_rows)

    if not metrics_df.empty:
        mean_columns = [
            "LSTM_Front_RMSE_N",
            "LSTM_Front_MAE_N",
            "LSTM_Rear_RMSE_N",
            "LSTM_Rear_MAE_N",
            "UKF_Front_RMSE_N",
            "UKF_Front_MAE_N",
            "UKF_Rear_RMSE_N",
            "UKF_Rear_MAE_N",
        ]

        print(f"\n[{split_name}] Mean metrics")
        print(metrics_df[mean_columns].mean())

    return metrics_df

In [ ]:
## Validation 시나리오 결과 출력 ##
validation_metrics_path = "results/validation_metrics.txt"

validation_metrics_df = evaluate_scenario_list(
    scenario_list=val_scenarios,
    split_name="Validation",
    metrics_path=validation_metrics_path,
    dataset=dataset,
    model=model,
    x_scaler=x_scaler,
    y_scaler=y_scaler,
    sequence_length=sequence_length,
    device=device
)

print("\nValidation metrics summary")
display(validation_metrics_df)
print(f"\nSaved: {validation_metrics_path}")


In [ ]:
## Test 시나리오 결과 출력 ##
# Test split은 학습 및 validation loss 기반 모델 선택에 사용하지 않은
# 최종 일반화 성능 평가용 시나리오입니다.

test_metrics_path = "results/test_metrics.txt"

test_metrics_df = evaluate_scenario_list(
    scenario_list=test_scenarios,
    split_name="Test",
    metrics_path=test_metrics_path,
    dataset=dataset,
    model=model,
    x_scaler=x_scaler,
    y_scaler=y_scaler,
    sequence_length=sequence_length,
    device=device
)

print("\nTest metrics summary")
display(test_metrics_df)
print(f"\nSaved: {test_metrics_path}")
